# Soccer Results Data Preparation and Enrichment

This notebook covers:
1. Historical data cleaning and feature engineering with Spark.
2. Current-season web scraping and schema alignment.
3. Export of prepared datasets for streaming and machine learning.

## Part 1
## Spark Session Initialization

Create a Spark session for historical dataset preparation.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("prepare_data") \
    .getOrCreate()

## Historical Data Loading

Load the historical CSV file and inspect the raw schema.

In [3]:
df = spark.read.csv(
    "/workspace/data/rim_championnat_results_2007-2025.csv",
    header=True,
    inferSchema=True
)

df.show()
df.printSchema()

+--------------------+----------+---------------+---------------+----------+----------+
|              season|      date|      home_team|      away_team|home_goals|away_goals|
+--------------------+----------+---------------+---------------+----------+----------+
|Championnat D1 20...|02.08.2008|  ASAC Concorde|    Nasr Sebkha|         2|         0|
|Championnat D1 20...|02.08.2008|         Police|       Tidjikja|         0|         1|
|Championnat D1 20...|01.08.2008|           Ksar|           SNIM|         1|         0|
|Championnat D1 20...|01.08.2008|Mauritel Mobile|          Armee|         0|         0|
|Championnat D1 20...|01.08.2008|  Tevragh-Zeina|      FC Trarza|         3|         0|
|Championnat D1 20...|29.07.2008|Garde Nationale|     Nouadhibou|         0|         0|
|Championnat D1 20...|26.07.2008|  ASAC Concorde|     Nouadhibou|         1|         0|
|Championnat D1 20...|26.07.2008|  Tevragh-Zeina|Mauritel Mobile|         0|         2|
|Championnat D1 20...|25.07.2008

## Historical Cleaning and Type Standardization

Handle missing values, cast numeric columns, and normalize date parsing.

In [4]:
from pyspark.sql.functions import coalesce, col, to_date, trim

df = df.dropna(subset=["season", "date", "home_team", "away_team", "home_goals", "away_goals"])

df = (
    df.withColumn("home_goals", col("home_goals").cast("int"))
      .withColumn("away_goals", col("away_goals").cast("int"))
      .withColumn("date_raw", trim(col("date").cast("string")))
      .withColumn(
          "date",
          coalesce(
              to_date(col("date_raw"), "dd.MM.yyyy"),
              to_date(col("date_raw"), "dd/MM/yyyy"),
              to_date(col("date_raw"), "yyyy-MM-dd"),
          ),
      )
      .drop("date_raw")
)

df = df.filter(col("date").isNotNull())

## Historical Feature Engineering

Create analytical columns: total goals, goal difference, result class, and match year.

In [5]:
from pyspark.sql.functions import when, year

df = df.withColumn("total_goals", col("home_goals") + col("away_goals")) \
       .withColumn("goal_difference", col("home_goals") - col("away_goals")) \
       .withColumn(
           "result",
           when(col("home_goals") > col("away_goals"), "home_win")
           .when(col("home_goals") < col("away_goals"), "away_win")
           .otherwise("draw")
       ) \
       .withColumn("match_year", year(col("date")))

## Historical Parquet Export and Validation

Write the prepared historical dataset to Parquet and verify the result.

In [6]:
df.write.mode("overwrite").parquet("/workspace/data/outputs/prepared/historical/")

In [7]:
df_check = spark.read.parquet("/workspace/data/outputs/prepared/historical/")
df_check.show()
df_check.printSchema()

+--------------------+----------+---------------+---------------+----------+----------+-----------+---------------+--------+----------+
|              season|      date|      home_team|      away_team|home_goals|away_goals|total_goals|goal_difference|  result|match_year|
+--------------------+----------+---------------+---------------+----------+----------+-----------+---------------+--------+----------+
|Championnat D1 20...|2008-08-02|  ASAC Concorde|    Nasr Sebkha|         2|         0|          2|              2|home_win|      2008|
|Championnat D1 20...|2008-08-02|         Police|       Tidjikja|         0|         1|          1|             -1|away_win|      2008|
|Championnat D1 20...|2008-08-01|           Ksar|           SNIM|         1|         0|          1|              1|home_win|      2008|
|Championnat D1 20...|2008-08-01|Mauritel Mobile|          Armee|         0|         0|          0|              0|    draw|      2008|
|Championnat D1 20...|2008-08-01|  Tevragh-Zeina

## Part2:Current Season Enrichment (Web Scraping)

Scrape completed matches, normalize schema and team names,and export the cleaned current-season CSV.

In [8]:
import os
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

SOURCE_URL = "http://www.soccerpunter.com/results/26572/Mauritania-Super-D1-2025-2026"
TARGET_SEASON = "2025-2026"
SEASON_START = pd.Timestamp("2025-08-01")
SEASON_END = pd.Timestamp("2026-07-31")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}
response = requests.get(SOURCE_URL, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

matches = []
for row in soup.select("tr.betMatchRow"):
    date_link = row.select_one("a.dateLink")
    home_link = row.select_one("td.teamHome a.teamLink")
    away_link = row.select_one("td.teamAway a.teamLink")

    if not date_link or not home_link or not away_link:
        continue

    row_text = " ".join(row.get_text(" ", strip=True).split())
    score_match = re.search(r"\b(\d+)\s*-\s*(\d+)\b", row_text)
    if not score_match:
        continue

    home_goals = int(score_match.group(1))
    away_goals = int(score_match.group(2))

    date_raw = date_link.get_text(strip=True)
    match_date = pd.to_datetime(date_raw, format="%d/%m/%Y", errors="coerce")
    if pd.isna(match_date):
        continue

    if not (SEASON_START <= match_date <= SEASON_END):
        continue

    matches.append(
        {
            "season": TARGET_SEASON,
            "date": match_date,
            "home_team": home_link.get_text(" ", strip=True),
            "away_team": away_link.get_text(" ", strip=True),
            "home_goals": home_goals,
            "away_goals": away_goals,
        }
    )

if not matches:
    raise ValueError("No 2025-2026 matches were collected from the selected source.")

scraped_df = pd.DataFrame(matches).drop_duplicates()

team_aliases = {
    "AC Douane": "Douane",
    "Police": "Police",
    "Kaedi": "Kaedi",
    "Tevragh-Zeïne": "Tevragh-Zeina",
    "Tevragh-Zeine": "Tevragh-Zeina",
    "Nouakchott Academie": "Nouakchott Academy",
    "N'Zidane": "Nzeidane",
}

scraped_df["home_team"] = scraped_df["home_team"].replace(team_aliases).str.strip()
scraped_df["away_team"] = scraped_df["away_team"].replace(team_aliases).str.strip()

scraped_df["total_goals"] = scraped_df["home_goals"] + scraped_df["away_goals"]
scraped_df["goal_difference"] = scraped_df["home_goals"] - scraped_df["away_goals"]
scraped_df["result"] = scraped_df.apply(
    lambda r: "home_win" if r["home_goals"] > r["away_goals"]
    else "away_win" if r["home_goals"] < r["away_goals"]
    else "draw",
    axis=1,
)
scraped_df["match_year"] = scraped_df["date"].dt.year
scraped_df["date"] = scraped_df["date"].dt.strftime("%Y-%m-%d")

scraped_df = scraped_df[
    [
        "season",
        "date",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals",
        "total_goals",
        "goal_difference",
        "result",
        "match_year",
    ]
].sort_values("date", ascending=False)

output_path = "/workspace/data/outputs/prepared/current_season"
os.makedirs(output_path, exist_ok=True)

csv_path = os.path.join(output_path, "current_season_clean.csv")
scraped_df.to_csv(csv_path, index=False)

print(scraped_df.head(10))
print(f"\nScraping completed ({len(scraped_df)} matches)")
print("Source:", SOURCE_URL)
print("Output:", csv_path)

      season        date           home_team           away_team  home_goals  \
0  2025-2026  2026-04-25               Kaedi          Nouadhibou           0   
1  2025-2026  2026-04-24                SNIM            Nzeidane           1   
2  2025-2026  2026-04-13             Gendrim       Tevragh-Zeina           1   
3  2025-2026  2026-04-12  Nouakchott Academy              Police           0   
4  2025-2026  2026-04-11              Chemal              Douane           1   
5  2025-2026  2026-04-11               Kaedi            Nzeidane           2   
6  2025-2026  2026-04-11                SNIM   Nouakchott King's           1   
7  2025-2026  2026-04-10          Nouadhibou                Ksar           5   
8  2025-2026  2026-04-09            Pompiers    Inter Nouakchott           3   
9  2025-2026  2026-04-07              Douane  Nouakchott Academy           1   

   away_goals  total_goals  goal_difference    result  match_year  
0           1            1               -1  away_w

## Unified Prepared Layer Export

Merge historical and current-season datasets into one consistent Parquet dataset.

In [9]:
current_spark = spark.createDataFrame(scraped_df).withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

common_cols = [
    "season",
    "date",
    "home_team",
    "away_team",
    "home_goals",
    "away_goals",
    "total_goals",
    "goal_difference",
    "result",
    "match_year",
]

merged_df = df.select(*common_cols).unionByName(current_spark.select(*common_cols))
merged_df.write.mode("overwrite").parquet("/workspace/data/outputs/prepared/merged/")

print("Merged dataset exported to /workspace/data/outputs/prepared/merged/")
print("Historical rows:", df.count())
print("Current season rows:", current_spark.count())
print("Merged rows:", merged_df.count())

Merged dataset exported to /workspace/data/outputs/prepared/merged/
Historical rows: 2590
Current season rows: 151
Merged rows: 2741
